# Clasificador de perros y gatos

Exploracion visual, preparacion del dataset, entrenamiento y evaluacion de EfficientNetB0.

**Nota para el tutor:** siguiendo el paso de preprocesamiento de la guia, tanto las imagenes como la entrada del modelo se ajustan a 200x200.

In [ ]:
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt

project_root = Path.cwd()
if project_root.name == "src":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from image_classifier import (
    BATCH_SIZE,
    IMAGE_SIZE,
    RAW_IMAGES_DIR,
    SEED,
    build_model,
    create_generators,
    evaluate_model,
    load_best_model,
    prepare_data,
    train_model,
)

print(f"Dataset: {RAW_IMAGES_DIR}")
print(f"Imagenes por entrada: {IMAGE_SIZE[0]} x {IMAGE_SIZE[1]}")

In [ ]:
class_images = {
    "cat": sorted(RAW_IMAGES_DIR.glob("cat.*.jpg")),
    "dog": sorted(RAW_IMAGES_DIR.glob("dog.*.jpg")),
}

for class_name, image_paths in class_images.items():
    print(f"{class_name}: {len(image_paths)} imagenes")
    selected_paths = random.Random(SEED).sample(image_paths, min(9, len(image_paths)))
    figure, axes = plt.subplots(3, 3, figsize=(9, 9))
    for axis, image_path in zip(axes.flat, selected_paths):
        axis.imshow(plt.imread(image_path))
        axis.axis("off")
    figure.suptitle(f"Ejemplos de {class_name}")
    plt.tight_layout()
    plt.show()

In [ ]:
train_dir, validation_dir, test_dir = prepare_data()
trdata, valdata, tsdata = create_generators(
    train_dir,
    validation_dir,
    test_dir,
    batch_size=BATCH_SIZE,
)

print("Clases detectadas:", trdata.class_indices)
print(f"Entrenamiento: {trdata.samples} imagenes")
print(f"Validacion: {valdata.samples} imagenes")
print(f"Prueba: {tsdata.samples} imagenes")

In [ ]:
model = build_model()
model.summary()

In [ ]:
history = train_model(model, trdata, valdata, epochs=20)

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, metric, title in zip(axes, ("accuracy", "loss"), ("Accuracy", "Loss")):
    axis.plot(history.history[metric], label="train")
    axis.plot(history.history[f"val_{metric}"], label="validation")
    axis.set_title(title)
    axis.set_xlabel("Epoca")
    axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_model = load_best_model()
test_metrics = evaluate_model(best_model, tsdata)
print("Metricas en test:", test_metrics)